In [ ]:
# py -m pip install git+https://github.com/pbtamarona/OpenPyTEA
from openpytea.equipment import *
from openpytea.plant import *
from openpytea.analysis import *
from openpytea.plotting import *

In [ ]:
#OPEX Input
country, region = 'Netherlands', 'None'
interest_rate, project_lifetime, plant_utilization = 0.09, 20, 0.95
h2_prod = 100_000  # Daily H₂ production in kg (100 tonnes/day)
h2_price = 5       # US$/kg
tax_rate = 0.258

operator_rate, operator_std, operator_max, operator_min = 39.04, 12.31, 55, 23.43
# Operator hourly rate for USA-Gulf Coast, in US$/h


elecs_price, elecs_std, elecs_max, elecs_min = 0.125, 0.05, 0.5, 0.05
# Price of electricity, in US$/kWh

nat_gas_price, nat_gas_std, nat_gas_max, nat_gas_min = 0.8646, 0.4, 4.884, 0.30261
# Price of natural gas, in US$/kg

cool_water_price, cool_water_std, cool_water_max, cool_water_min = 2.4592e-4, 1e-4, 4e-4, 1e-5
# Price of cooling water, in US$/MJ

co2_tax, co2_std, co2_max, co2_min = 0.10201, 0.025, 0.19, 0 #US$/kgCO2

In [ ]:
# PSA Sizing Function
import numpy as np

def calculate_psa_vessel_thickness(vessel_diameter, diameter_threshold=3.5):
    # Minimum practical wall thickness from Towler & Sinnott
    if vessel_diameter <= 1.0:
        minimum_thickness_mm = 5.0
    elif vessel_diameter <= 2.0:
        minimum_thickness_mm = 7.0
    elif vessel_diameter <= 2.5:
        minimum_thickness_mm = 9.0
    elif vessel_diameter <= 3.0:
        minimum_thickness_mm = 10.0
    else:
        minimum_thickness_mm = 12.0

    # Additional thickness margin
    if vessel_diameter <= diameter_threshold:
        extra_thickness_mm = 5.0
    else:
        extra_thickness_mm = 10.0

    return minimum_thickness_mm + extra_thickness_mm


def calculate_psa_vessel_dimensions(adsorbent_mass_per_bed, rho_bulk=795.0, inert_fraction=0.20, aspect_ratio=4.0):
    # Adsorbent volume per bed [m3]
    adsorbent_volume = adsorbent_mass_per_bed / rho_bulk

    # Total internal vessel volume between tangent lines [m3]
    vessel_volume = adsorbent_volume / (1 - inert_fraction)

    # Vessel geometry
    vessel_diameter = (4 * vessel_volume / (np.pi * aspect_ratio)) ** (1 / 3)
    vessel_height = aspect_ratio * vessel_diameter

    return {
        "adsorbent_volume": adsorbent_volume,
        "vessel_volume": vessel_volume,
        "vessel_diameter": vessel_diameter,
        "vessel_height": vessel_height,
        "aspect_ratio": aspect_ratio
    }

def calculate_psa_shell_mass(vessel_diameter, vessel_height, steel_density=7850.0, diameter_threshold=3.5):
    # Wall thickness from vessel diameter
    thickness_mm = calculate_psa_vessel_thickness(vessel_diameter, diameter_threshold=diameter_threshold)
    thickness = thickness_mm / 1000  # m

    # Internal and external diameters
    D_inner = vessel_diameter
    D_outer = D_inner + 2 * thickness

    # Steel volume of cylindrical shell [m3]
    steel_volume = np.pi / 4 * (D_outer**2 - D_inner**2) * vessel_height

    # Shell mass [kg]
    shell_mass = steel_volume * steel_density

    return {
        "wall_thickness_mm": thickness_mm,
        "steel_volume": steel_volume,
        "shell_mass": shell_mass
    }

def calculate_psa_design(psa_feed_total, ch4_feed_total, p_ads_bar=15.0, p_des_bar=1.01, adsorption_time=300.0, bed_loading_fraction=0.85, min_trains=2, n_beds_per_train=4, h2_purity=0.995, product_mass_flow=100_000, rho_bulk=795.0, inert_fraction=0.20, min_aspect_ratio=4.0, max_aspect_ratio=6.0, aspect_ratio_step=0.5, max_diameter=3.5, max_trains=15):

    MW_H2 = 2.01588    # kg/kmol
    MW_CH4 = 16.04246  # kg/kmol

    product_mass_total = product_mass_flow / 24  # kg/h

    # Product purity
    MW_product = h2_purity * MW_H2 + (1 - h2_purity) * MW_CH4

    # Total product molar flow and component molar flows [kmol/h]
    total_product_total = product_mass_total / MW_product
    h2_product_total = total_product_total * h2_purity
    ch4_product_total = total_product_total * (1 - h2_purity)

    # CH4 / ZEOLITE 5A ISOTHERM - Towler & Sinnott, Example 16.1
    pressure_data = np.array([0.0, 1.5, 2.5, 4.5, 6.5, 9.0, 12.5, 16.5, 19.5])
    loading_data = np.array([0.0, 0.4, 0.7, 1.0, 1.25, 1.45, 1.60, 1.70, 1.75])  # mmol/g = mol/kg

    bar_to_atm = 0.986923
    valid_design = False
    diameter_fallback = False

    # Number of trains and H/D ratio
    for n_trains in range(min_trains, max_trains + 1):

        # Flow handled by one PSA train
        psa_feed = psa_feed_total / n_trains
        ch4_feed = ch4_feed_total / n_trains
        h2_feed = psa_feed - ch4_feed

        # Product handled by one PSA train
        total_product = total_product_total / n_trains
        h2_product = h2_product_total / n_trains
        ch4_product = ch4_product_total / n_trains

        h2_recovery = h2_product / h2_feed

        # PSA off-gas
        h2_offgas = h2_feed - h2_product
        ch4_offgas = ch4_feed - ch4_product
        total_offgas = h2_offgas + ch4_offgas

        if h2_offgas < 0 or ch4_offgas < 0 or total_offgas <= 0:
            raise ValueError("Calculated PSA outlet flows are not physically consistent.")

        # CH4 mole fractions
        y_ch4_feed = ch4_feed / psa_feed
        y_ch4_offgas = ch4_offgas / total_offgas

        # CH4 partial pressures [atm]
        p_ch4_ads = p_ads_bar * y_ch4_feed * bar_to_atm
        p_ch4_des = p_des_bar * y_ch4_offgas * bar_to_atm

        if p_ch4_ads > pressure_data[-1]:
            raise ValueError("Adsorption CH4 partial pressure exceeds isotherm range.")

        if p_ch4_des > pressure_data[-1]:
            raise ValueError("Desorption CH4 partial pressure exceeds isotherm range.")

        # Equilibrium loadings
        q_ads = np.interp(p_ch4_ads, pressure_data, loading_data)
        q_des = np.interp(p_ch4_des, pressure_data, loading_data)
        working_capacity = q_ads - q_des

        if working_capacity <= 0:
            raise ValueError("PSA working capacity is zero or negative.")

        # ADSORBENT MASS PER BED - Towler Eq. 16.2
        ch4_removed = ch4_feed - ch4_product  # kmol/h
        ch4_removed_mol_s = ch4_removed * 1000 / 3600
        adsorbent_mass_per_bed = ch4_removed_mol_s * adsorption_time / (working_capacity * bed_loading_fraction)

        # Discrete H/D ratios
        aspect_ratios = np.arange(min_aspect_ratio, max_aspect_ratio + aspect_ratio_step / 2, aspect_ratio_step)

        for aspect_ratio in aspect_ratios:
            vessel = calculate_psa_vessel_dimensions(adsorbent_mass_per_bed, rho_bulk=rho_bulk, inert_fraction=inert_fraction, aspect_ratio=aspect_ratio)

            if vessel["vessel_diameter"] <= max_diameter:
                valid_design = True
                break

        # Fallback at max number of trains
        if not valid_design and n_trains == max_trains:
            aspect_ratio = max_aspect_ratio
            vessel = calculate_psa_vessel_dimensions(adsorbent_mass_per_bed, rho_bulk=rho_bulk, inert_fraction=inert_fraction, aspect_ratio=aspect_ratio)

            valid_design = True
            diameter_fallback = True

        if valid_design:
            break

    if not valid_design:
        raise ValueError("No valid PSA configuration could be determined.")

    # Total number of vessels and adsorbent mass
    n_vessels = n_trains * n_beds_per_train
    total_adsorbent_mass = adsorbent_mass_per_bed * n_vessels

    # Steel volume and shell mass
    shell = calculate_psa_shell_mass(vessel["vessel_diameter"], vessel["vessel_height"], diameter_threshold=max_diameter)
    total_shell_mass = shell["shell_mass"] * n_vessels

    return {
        "h2_recovery": h2_recovery,
        "total_product_molar_flow": total_product_total,
        "h2_product_molar_flow": h2_product_total,
        "ch4_product_molar_flow": ch4_product_total,
        "adsorbent_mass_per_bed": adsorbent_mass_per_bed,
        "total_adsorbent_mass": total_adsorbent_mass,
        "vessel_volume": vessel["vessel_volume"],
        "vessel_diameter": vessel["vessel_diameter"],
        "vessel_height": vessel["vessel_height"],
        "aspect_ratio": vessel["aspect_ratio"],
        "wall_thickness_mm": shell["wall_thickness_mm"],
        "steel_volume_per_vessel": shell["steel_volume"],
        "shell_mass_per_vessel": shell["shell_mass"],
        "total_shell_mass": total_shell_mass,
        "n_trains": n_trains,
        "n_beds_per_train": n_beds_per_train,
        "n_vessels": n_vessels,
        "diameter_fallback": diameter_fallback
    }

In [ ]:
# Equipment list
#Compressors
comp_1_pwr = 1974  # kW
comp_2_pwr = 964  # kW
comp_1 = Equipment('COMP-1', comp_1_pwr, "Fluids", "Compressors, fans, & blowers", "Compressor, centrifugal", "Carbon steel")
comp_2 = Equipment('COMP-2', comp_2_pwr, "Fluids", "Compressors, fans, & blowers", "Compressor, centrifugal", "Carbon steel")

# Electrical to mechanical efficiency
mtr_eff   = 0.9
mtr_1_pwr = comp_1_pwr / mtr_eff
mtr_2_pwr = comp_2_pwr / mtr_eff

#Correlation requires shaft power, here assumed in direct connection with shaft of the compressor so with negligible mechanical losses of coupling. There will be losses in electrical requirement
mtr_1 = Equipment('MTR-1', comp_1_pwr, "Electrical", "Motors & generators", "Totally enclosed motor", "Carbon steel")
mtr_2 = Equipment('MTR-2', comp_2_pwr, "Electrical", "Motors & generators", "Totally enclosed motor", "Carbon steel")

#Heat exchangers
hx_1   = Equipment('HX-1',   66.54,   "Fluids", "Heat Exchangers", "U-tube shell & tube", "304 stainless steel")
hx_2   = Equipment('HX-2',   47.04,   "Fluids", "Heat Exchangers", "U-tube shell & tube", "304 stainless steel")
hx_3   = Equipment('HX-3',   218.84,  "Fluids", "Heat Exchangers", "U-tube shell & tube", "304 stainless steel")
cooler = Equipment('COOLER',  424.68, "Fluids", "Heat Exchangers", "U-tube shell & tube", "Carbon steel")

#SMR and WGS reactors.
smr_reactor    = Equipment('SMR-REACTOR',    160,     "Fluids", "Reactors", "Tubular fixed bed", "304 stainless steel") #min 160 in correlation
wgs_ht_reactor = Equipment('WGS-HT-REACTOR', 376.54,  "Fluids", "Reactors", "Tubular fixed bed", "304 stainless steel")
wgs_lt_reactor = Equipment('WGS-LT-REACTOR', 3822.92, "Fluids", "Reactors", "Tubular fixed bed", "304 stainless steel")

#Remaining equipment.
furnace = Equipment('FURNACE', 69.22,   "Fluids", "Boilers, heaters, & furnaces", "Furnace, cylindrical",     "304 stainless steel")
pump    = Equipment('PUMP',    10.79,    "Fluids", "Pumps",                        "Single-stage centrifugal", "Carbon steel")

#PSA
#Process inputs
psa_feed_total = 4012.99   # Total PSA feed [kmol/h]
ch4_feed_total = 1741.45   # CH4 in PSA feed [kmol/h]

psa_design = calculate_psa_design(psa_feed_total=psa_feed_total, ch4_feed_total=ch4_feed_total, p_ads_bar=14.91, p_des_bar=1.2, min_trains=2, n_beds_per_train=4)

psa_shell_mass = psa_design["shell_mass_per_vessel"]
psa_adsorbent_mass_total = psa_design["total_adsorbent_mass"]
n_psa_vessels = psa_design["n_vessels"]

# PSA vessels
psa_vessel = Equipment(name="PSA-VESSEL", param=psa_shell_mass, process_type="Fluids", category="Pressure vessels", type="Vertical CS", material="Carbon steel", num_units=n_psa_vessels)

# Zeolite 5A
psa_adsorbent_volume_total_m3 = psa_adsorbent_mass_total / 795.0
psa_adsorbent_volume_total_ft3 = psa_adsorbent_volume_total_m3 * 35.3146667

psa_zeolite = Equipment(name="PSA-ZEOLITE-5A", param=psa_adsorbent_volume_total_ft3, process_type="Solids", category="Packings & adsorbents", type="Molecular sieves")

# Complete PSA unit
psa = CompositeEquipment(name="PSA", process_type="Fluids", components=[psa_vessel, psa_zeolite], category="Adsorption", type="Pressure-swing adsorber")

# We can now create a list of the equipment for the SMR process.
smr_equipment = [
    comp_1, mtr_1,
    comp_2, mtr_2,
    hx_1, hx_2, hx_3, cooler,
    smr_reactor, wgs_ht_reactor, wgs_lt_reactor,
    furnace, pump, psa
]

In [ ]:
# Configuration for the SMR plant
config = {
    'plant_name'          : 'Steam Reforming',
    'process_type'        : 'Fluids',
    'country'             : country,
    'region'              : region,
    'equipment'           : smr_equipment,
    'interest_rate'       : interest_rate,
    'project_lifetime'    : project_lifetime,
    'plant_utilization'   : plant_utilization,
    'tax_rate'            : tax_rate,
    'operator_hourly_rate': {
        'rate': operator_rate, 'rate_uncertainty': {'std': operator_std, 'max': operator_max, 'min': operator_min}
    },
    'plant_products': {
        'hydrogen': {
            'production': h2_prod,   # Daily production in kg/day
            'price'     : h2_price,  # Price of hydrogen, in US$/kg
        }
    },
    'variable_opex_inputs': {
        'electricity': {
            'consumption': (mtr_1_pwr + mtr_2_pwr)*24,
            'price': elecs_price, 'price_uncertainty': {'std': elecs_std, 'max': elecs_max, 'min': elecs_min}
        },
        'methane_feed': {
            'consumption': (10731.4+7778.21)*24, # Methane feed in kg/day (the value comes from the Aspen Plus simulation)
            'price': nat_gas_price, 'price_uncertainty': {'std': nat_gas_std, 'max': nat_gas_max, 'min': nat_gas_min}
        },
        'cooling_water': {
            'consumption': 46281.96*24, # Cooling duties in Cooler (the value comes from the Aspen Plus simulation in MJ/hr)
            'price': cool_water_price, 'price_uncertainty': {'std': cool_water_std, 'max': cool_water_max, 'min': cool_water_min}
        },
        'CO$_2$_tax': {
            'consumption': 50747.7*24, # CO₂ emissions in kg/day (the value comes from the Aspen Plus simulation)
            'price': co2_tax, 'price_uncertainty': {'std': co2_std, 'max': co2_max, 'min': co2_min}
        },
    },
}
smr_plant = Plant(config)

In [ ]:
smr_plant.calculate_fixed_capital(print_results=True)   # Calculate the FCI and print the details
smr_plant.calculate_variable_opex(print_results=True)   # Calculate the Variable OPEX and print the details
smr_plant.calculate_levelized_cost(print_results=True)  # Calculate the LCOH and print the details

In [ ]:
import matplotlib.pyplot as plt

#Samples
mc_smr = monte_carlo(
    smr_plant,
    num_samples=1_000_000,
    batch_size=10_000,
    random_seed=12345
)

#Plot
with plt.style.context("default"):

    fig, ax = plot_monte_carlo(
        mc_smr,
        metric="LCOP",
        bins=60,
        figsize=(8, 5),
        show=False
    )

    fig.set_dpi(120)
    fig.set_facecolor("white")
    ax.set_facecolor("white")

    ax.set_title("Monte Carlo LCOH - SMR", fontsize=14, pad=12)

    ax.set_xlabel(r"LCOH / [US\$ $\cdot$ kg$^{-1}$]", fontsize=13, labelpad=10)
    ax.set_ylabel("Probability density", fontsize=13, labelpad=8)

    ax.tick_params(
        axis="both", which="major", direction="in",
        top=True, right=True, labelsize=11, length=5
    )

    ax.tick_params(
        axis="both", which="minor", direction="in",
        top=True, right=True, length=3
    )

    ax.minorticks_on()

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="upper right", fontsize=11, frameon=True)

    fig.tight_layout()
    plt.show()

In [ ]:
# Save Monte Carlo results - SMR
import json
from pathlib import Path

# Find project root
try:
    start_dir = Path(__file__).resolve().parent
except NameError:
    start_dir = Path.cwd()

project_root = next(
    (p for p in [start_dir, *start_dir.parents]
     if all((p / folder).is_dir() for folder in ["SMR", "WaterElectrolysis"])),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root not found.")

# Output directory
results_dir = project_root / "SMR"
results_dir.mkdir(parents=True, exist_ok=True)

mc_file = results_dir / "LCOH_MonteCarlo_SMR.json"

# Prepare Monte Carlo results
mc_export = {
    "name": "SMR",
    "metrics": {
        "LCOP": mc_smr["metrics"]["LCOP"].tolist()
    }
}

# Save JSON
with open(mc_file, "w") as f:
    json.dump(mc_export, f, indent=4)

print(f"JSON saved in: {mc_file.resolve()}")